# HGTCM P2 - Target OCR Training and Prediction Export

This notebook trains the canonical target-only OCR model and writes the train/val/test prediction handoff used by BART correction and held-out scoring. The An Gaodhal branches remain archival/reference-only and are excluded from the canonical target study unless explicitly enabled.

## Imports


In [ ]:
# ## Light imports, display helpers, and OMP duplicate-lib shim for some stacks.
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

from pathlib import Path
import importlib.util

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 110)


def section(title):
    display(Markdown(f'### {title}'))


def summary_table(summary, title=None):
    if title:
        section(title)
    display(pd.DataFrame([summary]))


def training_summary_table(summary, title=None):
    if title:
        section(title)
    keys = ['model_name_or_path', 'train_samples', 'val_samples', 'epochs', 'device', 'best_val_cer', 'best_checkpoint']
    display(pd.DataFrame([{key: summary.get(key) for key in keys}]))



## Visible Code: TrOCR Training Utilities

Training logic is intentionally kept in this notebook instead of a hidden Python module.



In [ ]:
# ## TrOCR workspace: paths, CER/WER, env readers, and train_trocr / infer / checkpoint discovery.
# Why: CER is the primary optimization target because small character edits dominate OCR utility.

import os
import json
import math
import random
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image

@dataclass(frozen=True)
class Workspace:
    root: Path

    @property
    def angaodhal_page_zip(self) -> Path:
        return self.root / "AnGaodhal_transkribusPage.zip"

    @property
    def angaodhal_alto_zip(self) -> Path:
        return self.root / "AnGaodhal_alto.zip"

    @property
    def angaodhal_metadata_csv(self) -> Path:
        return self.root / "AnGaodhal_pageMetadata.csv"

    @property
    def target_gt_zip(self) -> Path:
        matches = sorted(self.root.glob("GroundTruth811-*.zip"))
        return matches[0] if matches else self.root / "GroundTruth811-20260425T073039Z-3-001.zip"

    @property
    def target_image_zips(self) -> list[Path]:
        return sorted(self.root.glob("drive-download-*.zip"))

    @property
    def data_dir(self) -> Path:
        return self.root / "data"

    @property
    def manifests_dir(self) -> Path:
        return self.root / "manifests"

    @property
    def runs_dir(self) -> Path:
        return self.root / "runs"

    @property
    def reports_dir(self) -> Path:
        return self.root / "reports"


def workspace_from(value: str | None = None) -> Workspace:
    return Workspace(Path(value or ".").resolve())


def utc_run_id(prefix: str) -> str:
    return f"{prefix}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"


def looks_like_local_model_path(value: str | Path) -> bool:
    raw = str(value).strip()
    if not raw:
        return False
    path = Path(raw)
    return path.exists() or raw.endswith('best_checkpoint') or raw.endswith('last_checkpoint') or 'checkpoint' in raw.lower()


def set_training_seed(seed: int | None, torch_module=None) -> None:
    if seed is None:
        return
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except Exception:
        pass
    if torch_module is not None:
        torch_module.manual_seed(seed)
        if torch_module.cuda.is_available():
            torch_module.cuda.manual_seed_all(seed)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path: Path, payload: object) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return path


def read_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl(path: Path, rows: list[dict]) -> Path:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return path


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def normalize_text(text: str) -> str:
    return " ".join((text or "").replace("\r\n", "\n").replace("\r", "\n").split())

def env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip().lower() in {"1", "true", "yes", "on"}


def env_int_or_none(name: str, default: int | None = None) -> int | None:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return default
    if raw.strip().lower() in {"none", "all", "full"}:
        return None
    return int(raw)



WIN_MANIFEST_PREFIX = "C:\\Users\\Jobias\\OneDrive\\Desktop\\Project Decode\\"
PROJECT_ROOT = Path(os.environ.get("PROJECT_DECODE_ROOT", Path.cwd())).resolve()


def portable_path(path_value: str | Path) -> Path:
    """Rewrite Windows-absolute manifest paths under PROJECT_DECODE_ROOT."""

    raw = str(path_value)
    if raw.startswith(WIN_MANIFEST_PREFIX):
        rel_parts = raw[len(WIN_MANIFEST_PREFIX):].split("\\")
        return PROJECT_ROOT.joinpath(*rel_parts)
    path = Path(raw)
    if path.exists():
        return path
    if not path.is_absolute():
        return PROJECT_ROOT / path
    parts = path.parts
    if "data" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("data"):])
    if "manifests" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("manifests"):])
    if "runs" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("runs"):])
    if "reports" in parts:
        return PROJECT_ROOT.joinpath(*parts[parts.index("reports"):])
    return path


def resolve_image_path(row: dict, workspace: Workspace | None = None) -> Path:
    image_path = portable_path(row["image_path"])
    if image_path.exists():
        return image_path
    raise FileNotFoundError(f"Could not resolve image path for {row.get('sample_id')}: {row.get('image_path')} -> {image_path}")
def levenshtein(left: list | str, right: list | str) -> int:
    if left == right:
        return 0
    if not left:
        return len(right)
    if not right:
        return len(left)
    previous = list(range(len(right) + 1))
    for left_index, left_item in enumerate(left, start=1):
        current = [left_index]
        for right_index, right_item in enumerate(right, start=1):
            current.append(min(current[right_index - 1] + 1, previous[right_index] + 1, previous[right_index - 1] + (left_item != right_item)))
        previous = current
    return previous[-1]


def cer(reference: str, prediction: str) -> float:
    return levenshtein(reference or "", prediction or "") / max(1, len(reference or ""))


def wer(reference: str, prediction: str) -> float:
    ref_words = normalize_text(reference).split()
    pred_words = normalize_text(prediction).split()
    return levenshtein(ref_words, pred_words) / max(1, len(ref_words))


# ## train_trocr: tokenized line crops, AdamW loop, val CER, best checkpoint.

def train_trocr(
    *,
    manifest_path: Path,
    output_dir: Path,
    model_name_or_path: str,
    epochs: int = 2,
    batch_size: int = 2,
    gradient_accumulation_steps: int = 4,
    learning_rate: float = 5e-5,
    warmup_ratio: float = 0.1,
    lr_scheduler_type: str = "cosine",
    max_target_length: int = 256,
    max_train_samples: int | None = None,
    max_val_samples: int | None = None,
    device: str = "auto",
    dataloader_num_workers: int = 0,
    dataloader_pin_memory: bool = False,
    precision: str = "auto",
    run_name: str | None = None,
    use_wandb: bool = False,
    wandb_project: str = "project-decode",
    logging_steps: int = 25,
    max_optimizer_steps: int | None = None,
    early_stopping_patience: int | None = 2,
    early_stopping_min_delta: float = 0.001,
    sanity_eval_after_n_evals: int = 3,
    sanity_max_eval_cer: float = 30.0,
    sanity_max_loss_multiple: float = 5.0,
    wandb_alerts: bool = True,
    seed: int | None = 42,
    allow_warm_start: bool = False,
) -> dict:
    torch, DataLoader, Dataset, TrOCRProcessor, VisionEncoderDecoderModel = _require_training_deps()
    if looks_like_local_model_path(model_name_or_path) and not allow_warm_start:
        raise SystemExit(f"Refusing canonical TrOCR warm start from local path: {model_name_or_path}. Set ALLOW_CANONICAL_WARM_START=1 only for archival comparisons.")
    set_training_seed(seed, torch)
    rows = read_jsonl(manifest_path)
    train_rows = _select_rows(rows, "train", max_train_samples)
    val_rows = _select_rows(rows, "val", max_val_samples) or train_rows[: min(8, len(train_rows))]
    if not train_rows:
        raise SystemExit(f"No train rows found in {manifest_path}")
    processor = TrOCRProcessor.from_pretrained(model_name_or_path)
    model = VisionEncoderDecoderModel.from_pretrained(model_name_or_path)
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.sep_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id
    model.generation_config.max_length = max_target_length
    selected_device = "cuda" if device == "auto" and torch.cuda.is_available() else "cpu" if device == "auto" else device
    if selected_device.startswith("cuda"):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    model.to(selected_device)
    if precision == "auto":
        active_precision = "bf16" if selected_device.startswith("cuda") and torch.cuda.is_bf16_supported() else "no"
    else:
        active_precision = precision
    amp_enabled = selected_device.startswith("cuda") and active_precision in {"bf16", "fp16"}
    amp_dtype = torch.bfloat16 if active_precision == "bf16" else torch.float16
    run_name = run_name or output_dir.name
    started_at = time.time()
    wandb_run = None
    wandb_url = None
    if use_wandb:
        try:
            import wandb
            wandb_run = wandb.init(
                project=wandb_project,
                name=run_name,
                config={
                    "model_name_or_path": model_name_or_path,
                    "manifest_path": str(manifest_path),
                    "epochs": epochs,
                    "batch_size": batch_size,
                    "gradient_accumulation_steps": gradient_accumulation_steps,
                    "learning_rate": learning_rate,
                    "warmup_ratio": warmup_ratio,
                    "lr_scheduler_type": lr_scheduler_type,
                    "max_target_length": max_target_length,
                    "max_train_samples": max_train_samples,
                    "max_val_samples": max_val_samples,
                    "device": selected_device,
                    "dataloader_num_workers": dataloader_num_workers,
                    "dataloader_pin_memory": dataloader_pin_memory,
                    "precision": precision,
                    "max_optimizer_steps": max_optimizer_steps,
                    "early_stopping_patience": early_stopping_patience,
                    "early_stopping_min_delta": early_stopping_min_delta,
                    "sanity_eval_after_n_evals": sanity_eval_after_n_evals,
                    "sanity_max_eval_cer": sanity_max_eval_cer,
                    "sanity_max_loss_multiple": sanity_max_loss_multiple,
                },
            )
            wandb_url = wandb_run.get_url()
        except Exception as exc:
            print(f"[wandb] disabled for this run: {exc}")
            wandb_run = None
            wandb_url = None

    class OCRDataset(Dataset):
        def __init__(self, data: list[dict]):
            self.data = data

        def __len__(self) -> int:
            return len(self.data)

        def __getitem__(self, idx: int) -> dict:
            row = self.data[idx]
            image = Image.open(resolve_image_path(row, WORKSPACE)).convert("RGB")
            pixel_values = processor(image, return_tensors="pt").pixel_values.squeeze(0)
            labels = processor.tokenizer(row["normalized_text"], padding="max_length", max_length=max_target_length, truncation=True).input_ids
            labels = [label if label != processor.tokenizer.pad_token_id else -100 for label in labels]
            return {"pixel_values": pixel_values, "labels": torch.tensor(labels), "row": row}

    def collate(batch: list[dict]) -> dict:
        return {"pixel_values": torch.stack([item["pixel_values"] for item in batch]), "labels": torch.stack([item["labels"] for item in batch]), "rows": [item["row"] for item in batch]}

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    loader_kwargs = {
        "num_workers": dataloader_num_workers,
        "pin_memory": bool(dataloader_pin_memory and selected_device.startswith("cuda")),
    }
    if dataloader_num_workers > 0:
        loader_kwargs["persistent_workers"] = True
    train_loader = DataLoader(OCRDataset(train_rows), batch_size=batch_size, shuffle=True, collate_fn=collate, **loader_kwargs)
    val_loader = DataLoader(OCRDataset(val_rows), batch_size=batch_size, shuffle=False, collate_fn=collate, **loader_kwargs)
    optimizer_steps_per_epoch = max(1, math.ceil(len(train_loader) / max(1, gradient_accumulation_steps)))
    planned_optimizer_steps = max(1, epochs * optimizer_steps_per_epoch)
    step_cap_will_truncate = bool(max_optimizer_steps and max_optimizer_steps < planned_optimizer_steps)
    total_optimizer_steps = min(planned_optimizer_steps, max_optimizer_steps) if max_optimizer_steps else planned_optimizer_steps
    warmup_steps = int(total_optimizer_steps * warmup_ratio)
    completed_optimizer_steps = 0

    def current_lr(step_index: int) -> float:
        if lr_scheduler_type == "constant" or total_optimizer_steps <= 1:
            return learning_rate
        if warmup_steps > 0 and step_index <= warmup_steps:
            return learning_rate * (step_index / max(1, warmup_steps))
        if lr_scheduler_type == "cosine":
            progress = (step_index - warmup_steps) / max(1, total_optimizer_steps - warmup_steps)
            progress = min(1.0, max(0.0, progress))
            return learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))
        return learning_rate

    def set_optimizer_lr(lr: float) -> None:
        for group in optimizer.param_groups:
            group["lr"] = lr
    ensure_dir(output_dir)
    history: list[dict] = []
    best_val_cer = math.inf
    stale_epochs = 0
    eval_count = 0
    first_loss: float | None = None
    stop_training = False
    stop_reason = "completed"
    best_dir = output_dir / "best_checkpoint"
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        step_count = 0
        for step, batch in enumerate(train_loader, start=1):
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_enabled):
                outputs = model(pixel_values=batch["pixel_values"].to(selected_device, non_blocking=True), labels=batch["labels"].to(selected_device, non_blocking=True))
            raw_loss_value = float(outputs.loss.detach().cpu())
            if math.isnan(raw_loss_value) or math.isinf(raw_loss_value):
                stop_reason = f"sanity_stop_nan_or_inf_loss_epoch_{epoch}_step_{step}"
                stop_training = True
                print(f"[SanityKill] NaN/inf loss at epoch {epoch}, step {step}; stopping.")
                break
            if first_loss is None:
                first_loss = raw_loss_value
            elif raw_loss_value > first_loss * sanity_max_loss_multiple:
                stop_reason = f"sanity_stop_loss_explosion_epoch_{epoch}_step_{step}"
                stop_training = True
                print(f"[SanityKill] Loss {raw_loss_value:.4f} exceeds {sanity_max_loss_multiple}x starting loss {first_loss:.4f}; stopping.")
                break
            loss = outputs.loss / max(1, gradient_accumulation_steps)
            loss.backward()
            running_loss += raw_loss_value
            step_count += 1
            if step % gradient_accumulation_steps == 0 or step == len(train_loader):
                completed_optimizer_steps += 1
                set_optimizer_lr(current_lr(completed_optimizer_steps))
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if wandb_run is not None and (completed_optimizer_steps == 1 or completed_optimizer_steps % max(1, logging_steps) == 0):
                    wandb_run.log({
                        "train/loss_step": raw_loss_value,
                        "train/learning_rate": optimizer.param_groups[0]["lr"],
                        "train/optimizer_step": completed_optimizer_steps,
                        "epoch": epoch,
                    }, step=completed_optimizer_steps)
                if max_optimizer_steps and completed_optimizer_steps >= max_optimizer_steps:
                    stop_reason = f"max_optimizer_steps_{max_optimizer_steps}_reached"
                    stop_training = True
                    print(f"[BudgetStop] Reached max_optimizer_steps={max_optimizer_steps}; stopping after evaluation.")
                    break
        val_cer = _evaluate_model_cer(model, processor, val_loader, torch, selected_device, max_target_length)
        eval_count += 1
        train_loss = running_loss / max(1, step_count)
        history_row = {"epoch": epoch, "train_loss": train_loss, "val_cer": val_cer, "learning_rate": optimizer.param_groups[0]["lr"], "optimizer_steps": completed_optimizer_steps}
        history.append(history_row)
        if wandb_run is not None:
            wandb_run.log({
                "epoch": epoch,
                "train/loss_epoch": train_loss,
                "eval/cer": val_cer,
                "train/learning_rate": optimizer.param_groups[0]["lr"],
                "train/optimizer_step": completed_optimizer_steps,
            }, step=completed_optimizer_steps)
        previous_best = best_val_cer
        if val_cer < best_val_cer:
            best_val_cer = val_cer
            ensure_dir(best_dir)
            model.save_pretrained(best_dir)
            processor.save_pretrained(best_dir)
        if val_cer < previous_best - early_stopping_min_delta:
            stale_epochs = 0
        else:
            stale_epochs += 1
        if eval_count >= sanity_eval_after_n_evals and val_cer > sanity_max_eval_cer:
            stop_reason = f"sanity_stop_eval_cer_{val_cer:.4f}_after_{eval_count}_evals"
            stop_training = True
            print(f"[SanityKill] After {eval_count} evals, val_cer={val_cer:.4f} > {sanity_max_eval_cer}; stopping.")
            if wandb_run is not None and wandb_alerts:
                try:
                    import wandb
                    wandb.alert(title="Project Decode: val_cer high", text=f"{run_name}: val_cer={val_cer:.4f} after {eval_count} evals", level=wandb.AlertLevel.WARN)
                except Exception as exc:
                    print(f"[wandb] alert skipped: {exc}")
        if early_stopping_patience is not None and stale_epochs >= early_stopping_patience:
            stop_reason = f"early_stopping_patience_{early_stopping_patience}_reached"
            stop_training = True
            print(f"[EarlyStop] No CER improvement >= {early_stopping_min_delta} for {stale_epochs} evals; stopping.")
        if stop_training:
            break
    last_dir = output_dir / "last_checkpoint"
    ensure_dir(last_dir)
    model.save_pretrained(last_dir)
    processor.save_pretrained(last_dir)
    summary = {
        "manifest_path": str(manifest_path),
        "model_name_or_path": model_name_or_path,
        "output_dir": str(output_dir),
        "best_checkpoint": str(best_dir),
        "last_checkpoint": str(last_dir),
        "epochs": epochs,
        "train_samples": len(train_rows),
        "val_samples": len(val_rows),
        "device": selected_device,
        "precision": active_precision,
        "mixed_precision_enabled": amp_enabled,
        "dataloader_num_workers": dataloader_num_workers,
        "dataloader_pin_memory": bool(dataloader_pin_memory and selected_device.startswith("cuda")),
        "learning_rate": learning_rate,
        "warmup_ratio": warmup_ratio,
        "lr_scheduler_type": lr_scheduler_type,
        "seed": seed,
        "allow_warm_start": allow_warm_start,
        "step_cap_will_truncate": step_cap_will_truncate,
        "run_name": run_name,
        "planned_optimizer_steps": planned_optimizer_steps,
        "total_optimizer_steps": total_optimizer_steps,
        "completed_optimizer_steps": completed_optimizer_steps,
        "warmup_steps": warmup_steps,
        "best_val_cer": best_val_cer,
        "history": history,
        "logging_steps": logging_steps,
        "max_optimizer_steps": max_optimizer_steps,
        "early_stopping_patience": early_stopping_patience,
        "early_stopping_min_delta": early_stopping_min_delta,
        "sanity_eval_after_n_evals": sanity_eval_after_n_evals,
        "sanity_max_eval_cer": sanity_max_eval_cer,
        "sanity_max_loss_multiple": sanity_max_loss_multiple,
        "stop_reason": stop_reason,
        "elapsed_seconds": time.time() - started_at,
        "wandb_enabled": wandb_run is not None,
        "wandb_project": wandb_project if wandb_run is not None else None,
        "wandb_url": wandb_url,
    }
    write_json(output_dir / "train_summary.json", summary)
    if wandb_run is not None:
        if wandb_alerts:
            try:
                import wandb
                wandb.alert(title="Project Decode: run finished", text=f"{run_name}: best_val_cer={best_val_cer:.4f}, stop_reason={stop_reason}", level=wandb.AlertLevel.INFO)
            except Exception as exc:
                print(f"[wandb] finish alert skipped: {exc}")
        wandb_run.finish()
    return summary


# ## _require_training_deps: lazy import torch + TrOCR so the notebook still opens without them.

def _require_training_deps():
    try:
        import torch
        from torch.utils.data import DataLoader, Dataset
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    except ImportError as exc:
        raise SystemExit("Install training dependencies with: python -m pip install -r requirements-trocr.txt") from exc
    return torch, DataLoader, Dataset, TrOCRProcessor, VisionEncoderDecoderModel


def _is_trainable_crop(row: dict) -> bool:
    return row.get("crop_quality", {}).get("status", "ok") == "ok"


def _select_rows(rows: list[dict], split: str | None, limit: int | None) -> list[dict]:
    selected = [row for row in rows if (not split or row.get("dataset_split") == split) and _is_trainable_crop(row)]
    return selected[:limit] if limit else selected


def _evaluate_model_cer(model, processor, loader, torch, device: str, max_target_length: int) -> float:
    model.eval()
    scores: list[float] = []
    with torch.inference_mode():
        for batch in loader:
            generated_ids = model.generate(batch["pixel_values"].to(device, non_blocking=True), max_length=max_target_length)
            texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for row, prediction in zip(batch["rows"], texts):
                scores.append(cer(row["normalized_text"], prediction))
    return sum(scores) / max(1, len(scores))





# ## infer_trocr_predictions: load checkpoint, run batches, write predictions jsonl.

def infer_trocr_predictions(
    *,
    manifest_path: Path,
    model_dir: Path,
    output_path: Path,
    splits: tuple[str, ...] = ("train", "val", "test"),
    batch_size: int = 4,
    max_samples_per_split: int | None = None,
    device: str = "auto",
    max_target_length: int = 256,
    source_ocr_run_id: str | None = None,
) -> dict:
    torch, DataLoader, Dataset, TrOCRProcessor, VisionEncoderDecoderModel = _require_training_deps()
    rows_all = read_jsonl(manifest_path)
    selected_rows: list[dict] = []
    for split in splits:
        selected_rows.extend(_select_rows(rows_all, split, max_samples_per_split))
    if not selected_rows:
        raise SystemExit(f"No rows selected for OCR inference from {manifest_path}")

    processor = TrOCRProcessor.from_pretrained(str(model_dir))
    model = VisionEncoderDecoderModel.from_pretrained(str(model_dir))
    selected_device = "cuda" if device == "auto" and torch.cuda.is_available() else "cpu" if device == "auto" else device
    if selected_device.startswith("cuda"):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    model.to(selected_device)
    model.eval()

    class PredictionDataset(Dataset):
        def __init__(self, data: list[dict]):
            self.data = data

        def __len__(self) -> int:
            return len(self.data)

        def __getitem__(self, idx: int) -> dict:
            row = self.data[idx]
            image = Image.open(resolve_image_path(row, WORKSPACE)).convert("RGB")
            pixel_values = processor(image, return_tensors="pt").pixel_values.squeeze(0)
            return {"pixel_values": pixel_values, "row": row}

    def collate(batch: list[dict]) -> dict:
        return {"pixel_values": torch.stack([item["pixel_values"] for item in batch]), "rows": [item["row"] for item in batch]}

    loader = DataLoader(PredictionDataset(selected_rows), batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=2 if selected_device.startswith("cuda") else 0, pin_memory=selected_device.startswith("cuda"))
    prediction_rows: list[dict] = []
    with torch.inference_mode():
        for batch in loader:
            generated_ids = model.generate(batch["pixel_values"].to(selected_device), max_length=max_target_length)
            texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
            for row, prediction in zip(batch["rows"], texts):
                prediction = normalize_text(prediction)
                reference = normalize_text(row.get("normalized_text") or row.get("raw_text") or "")
                prediction_rows.append({
                    "sample_id": row.get("sample_id"),
                    "page_id": row.get("page_id"),
                    "group_id": row.get("group_id"),
                    "line_id": row.get("line_id"),
                    "dataset_split": row.get("dataset_split"),
                    "model_name": "target_trocr_raw",
                    "source_model_name": "target_only_trocr",
                    "source_ocr_run_id": source_ocr_run_id or model_dir.parent.name,
                    "source_checkpoint_path": str(model_dir),
                    "source_manifest_path": str(manifest_path),
                    "image_path": row.get("image_path"),
                    "prediction_raw": prediction,
                    "prediction_normalized": prediction,
                    "reference_raw": row.get("raw_text"),
                    "reference_normalized": reference,
                    "raw_cer": cer(reference, prediction),
                    "raw_wer": wer(reference, prediction),
                    "status": "ok",
                    "warnings": [],
                })
    write_jsonl(output_path, prediction_rows)
    split_counts = {split: sum(1 for row in prediction_rows if row.get("dataset_split") == split) for split in splits}
    summary = {
        "manifest_path": str(manifest_path),
        "model_dir": str(model_dir),
        "output_path": str(output_path),
        "source_ocr_run_id": source_ocr_run_id or model_dir.parent.name,
        "sample_count": len(prediction_rows),
        "split_counts": split_counts,
        "device": selected_device,
        "max_samples_per_split": max_samples_per_split,
    }
    write_json(output_path.with_suffix(".summary.json"), summary)
    return summary


## Workspace Configuration


In [ ]:
WORKSPACE = workspace_from(os.environ.get('PROJECT_DECODE_ROOT', '.'))
DATA_RUN_LABEL = os.environ.get('DATA_RUN_LABEL', 'affine_edge_full_clamped').strip() or 'affine_edge_full_clamped'
MODEL_NAME = os.environ.get('OCR_MODEL_NAME', 'microsoft/trocr-base-printed').strip() or 'microsoft/trocr-base-printed'
RUN_TRAINING = env_flag('RUN_TRAINING', '1')
RUN_TARGET_ONLY_TRAINING = env_flag('RUN_TARGET_ONLY_TRAINING', '1' if RUN_TRAINING else '0')
RUN_ANGAODHAL_REFERENCE_TRAINING = env_flag('RUN_ANGAODHAL_REFERENCE_TRAINING', '0')
RUN_TRANSFER_TRAINING = env_flag('RUN_TRANSFER_TRAINING', '0')
ALLOW_CANONICAL_WARM_START = env_flag('ALLOW_CANONICAL_WARM_START', '0')
WRITE_CANONICAL_TARGET_PREDICTIONS = env_flag('WRITE_CANONICAL_TARGET_PREDICTIONS', '1')
PREDICTION_MANIFEST_PATH = os.environ.get('PREDICTION_MANIFEST_PATH', '').strip()
PREDICTION_SPLITS = tuple(split.strip() for split in os.environ.get('PREDICTION_SPLITS', 'train,val,test').split(',') if split.strip()) or ('train', 'val', 'test')

summary_table(
    {
        'Workspace': str(WORKSPACE.root),
        'Data run label': DATA_RUN_LABEL,
        'Model': MODEL_NAME,
        'Run target-only training': RUN_TARGET_ONLY_TRAINING,
        'Run archive An Gaodhal reference training': RUN_ANGAODHAL_REFERENCE_TRAINING,
        'Run archive transfer training': RUN_TRANSFER_TRAINING,
        'Allow canonical warm start': ALLOW_CANONICAL_WARM_START,
        'Write canonical target predictions': WRITE_CANONICAL_TARGET_PREDICTIONS,
        'Prediction manifest override': PREDICTION_MANIFEST_PATH or None,
        'Prediction splits': ', '.join(PREDICTION_SPLITS),
    },
    'Training selection',
)


## Check Training Dependencies


In [ ]:
deps = ['torch', 'transformers', 'datasets', 'accelerate', 'PIL', 'cv2', 'pandas', 'tqdm', 'sentencepiece', 'wandb']
dep_status = {name: bool(importlib.util.find_spec(name)) for name in deps}
display(pd.DataFrame([{'Package': key, 'Installed': value} for key, value in dep_status.items()]))
if not all(dep_status.values()):
    display(Markdown('Install missing dependencies with `python -m pip install -r requirements-trocr.txt`.'))



## Load Training / Validation / Test Data


In [ ]:
target_manifest = WORKSPACE.manifests_dir / f'target_lines_{DATA_RUN_LABEL}.jsonl'
angaodhal_manifest = WORKSPACE.manifests_dir / f'angaodhal_lines_{DATA_RUN_LABEL}.jsonl'

target_rows = read_jsonl(target_manifest)
angaodhal_rows = read_jsonl(angaodhal_manifest) if RUN_ANGAODHAL_REFERENCE_TRAINING and angaodhal_manifest.exists() else []

manifest_summary = {
    'Target manifest': str(target_manifest),
    'Target rows': len(target_rows),
    'Target trainable rows': sum(1 for row in target_rows if _is_trainable_crop(row)),
}
if RUN_ANGAODHAL_REFERENCE_TRAINING:
    manifest_summary.update({
        'Archive An Gaodhal reference manifest': str(angaodhal_manifest),
        'Archive An Gaodhal reference rows': len(angaodhal_rows),
        'Archive An Gaodhal reference trainable rows': sum(1 for row in angaodhal_rows if _is_trainable_crop(row)),
    })

summary_table(manifest_summary, 'Input manifests')
if target_rows:
    display(pd.DataFrame(target_rows)['dataset_split'].value_counts())
if RUN_ANGAODHAL_REFERENCE_TRAINING and angaodhal_rows:
    display(pd.DataFrame(angaodhal_rows)['dataset_split'].value_counts())


## Training Utility Defaults


In [ ]:
# ## Training profile flags: smoke vs long runs, device batch sizes, cloud-oriented toggles.
# Why: profile toggles control compute budget while keeping the same data contract and metrics.
TRAINING_PROFILE = os.environ.get('TRAINING_PROFILE', 'smoke').strip().lower()
USE_CLOUD_ACCEL = TRAINING_PROFILE in {'cloud_2hr', 'cloud', 'h100', 'h200', 'full', 'gpu'}
USE_H200 = TRAINING_PROFILE in {'h200', 'full'}

def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))

TRAINING_DEFAULTS = {
    'profile': TRAINING_PROFILE,
    'model': MODEL_NAME,
    'epochs': env_int_or_none('OCR_EPOCHS', 5),
    'angaodhal_epochs': env_int_or_none('ANGAODHAL_EPOCHS', 14 if USE_H200 else 12 if USE_CLOUD_ACCEL else 5),
    'target_epochs': env_int_or_none('TARGET_EPOCHS', 10 if USE_H200 else 8 if USE_CLOUD_ACCEL else 5),
    'transfer_epochs': env_int_or_none('TRANSFER_EPOCHS', 10 if USE_H200 else 8 if USE_CLOUD_ACCEL else 5),
    'batch_size': env_int_or_none('OCR_BATCH_SIZE', 32 if USE_H200 else 16 if USE_CLOUD_ACCEL else 1),
    'gradient_accumulation_steps': env_int_or_none('OCR_GRADIENT_ACCUMULATION', 1 if USE_CLOUD_ACCEL else 8),
    'learning_rate': env_float('OCR_LEARNING_RATE', 5e-5),
    'warmup_ratio': env_float('OCR_WARMUP_RATIO', 0.1),
    'lr_scheduler_type': os.environ.get('OCR_LR_SCHEDULER', 'cosine'),
    'max_target_length': env_int_or_none('OCR_MAX_TARGET_LENGTH', 160 if USE_H200 else 128 if USE_CLOUD_ACCEL else 256),
    'max_train_samples': env_int_or_none('MAX_TRAIN_SAMPLES', None if USE_CLOUD_ACCEL else 32),
    'max_val_samples': env_int_or_none('MAX_VAL_SAMPLES', None if USE_CLOUD_ACCEL else 16),
    'max_predict_samples_per_split': env_int_or_none('MAX_PREDICT_SAMPLES_PER_SPLIT', None if USE_CLOUD_ACCEL else 16),
    'prediction_batch_size': env_int_or_none('OCR_PREDICTION_BATCH_SIZE', 64 if USE_H200 else 32 if USE_CLOUD_ACCEL else 4),
    'device': os.environ.get('OCR_DEVICE', 'auto'),
    'dataloader_num_workers': env_int_or_none('OCR_DATALOADER_WORKERS', 8 if USE_H200 else 4 if USE_CLOUD_ACCEL else 0),
    'dataloader_pin_memory': env_flag('OCR_DATALOADER_PIN_MEMORY', '1' if USE_CLOUD_ACCEL else '0'),
    'precision': os.environ.get('OCR_PRECISION', 'auto'),
    'use_wandb': (os.environ.get('USE_WANDB', '1' if USE_CLOUD_ACCEL else '0').strip().lower() in {'1', 'true', 'yes', 'on'}),
    'wandb_project': os.environ.get('WANDB_PROJECT', 'project-decode'),
    'logging_steps': env_int_or_none('OCR_LOGGING_STEPS', 25),
    'max_optimizer_steps': env_int_or_none('MAX_OPTIMIZER_STEPS', None),
    'early_stopping_patience': env_int_or_none('EARLY_STOPPING_PATIENCE', 4),
    'early_stopping_min_delta': env_float('EARLY_STOPPING_MIN_DELTA', 0.0005),
    'sanity_eval_after_n_evals': env_int_or_none('SANITY_EVAL_AFTER_N_EVALS', 3),
    'sanity_max_eval_cer': env_float('SANITY_MAX_EVAL_CER', 30.0),
    'sanity_max_loss_multiple': env_float('SANITY_MAX_LOSS_MULTIPLE', 5.0),
    'wandb_alerts': env_flag('WANDB_ALERTS', '1'),
    'seed': env_int_or_none('OCR_SEED', 42),
}
summary_table(TRAINING_DEFAULTS, f"{TRAINING_PROFILE} training defaults")


## Archive Only - An Gaodhal Reference OCR (Excluded From Canonical Target Study)

In [ ]:
# ## Optional archive/reference run: preserve the donor branch only when explicitly requested.
angaodhal_run_dir = WORKSPACE.runs_dir / utc_run_id(f'trocr_angaodhal_{DATA_RUN_LABEL}')

if RUN_ANGAODHAL_REFERENCE_TRAINING and angaodhal_rows:
    angaodhal_summary = train_trocr(
        manifest_path=angaodhal_manifest,
        output_dir=angaodhal_run_dir,
        model_name_or_path=MODEL_NAME,
        epochs=TRAINING_DEFAULTS['angaodhal_epochs'],
        batch_size=TRAINING_DEFAULTS['batch_size'],
        gradient_accumulation_steps=TRAINING_DEFAULTS['gradient_accumulation_steps'],
        learning_rate=TRAINING_DEFAULTS['learning_rate'],
        max_train_samples=TRAINING_DEFAULTS['max_train_samples'],
        max_val_samples=TRAINING_DEFAULTS['max_val_samples'],
        max_target_length=TRAINING_DEFAULTS['max_target_length'],
        warmup_ratio=TRAINING_DEFAULTS['warmup_ratio'],
        lr_scheduler_type=TRAINING_DEFAULTS['lr_scheduler_type'],
        device=TRAINING_DEFAULTS['device'],
        dataloader_num_workers=TRAINING_DEFAULTS['dataloader_num_workers'],
        dataloader_pin_memory=TRAINING_DEFAULTS['dataloader_pin_memory'],
        precision=TRAINING_DEFAULTS['precision'],
        run_name=angaodhal_run_dir.name,
        use_wandb=TRAINING_DEFAULTS['use_wandb'],
        wandb_project=TRAINING_DEFAULTS['wandb_project'],
        logging_steps=TRAINING_DEFAULTS['logging_steps'],
        max_optimizer_steps=TRAINING_DEFAULTS['max_optimizer_steps'],
        early_stopping_patience=TRAINING_DEFAULTS['early_stopping_patience'],
        early_stopping_min_delta=TRAINING_DEFAULTS['early_stopping_min_delta'],
        sanity_eval_after_n_evals=TRAINING_DEFAULTS['sanity_eval_after_n_evals'],
        sanity_max_eval_cer=TRAINING_DEFAULTS['sanity_max_eval_cer'],
        sanity_max_loss_multiple=TRAINING_DEFAULTS['sanity_max_loss_multiple'],
        wandb_alerts=TRAINING_DEFAULTS['wandb_alerts'],
    )
    training_summary_table(angaodhal_summary, 'Archive Archive An Gaodhal Reference Summary')
else:
    angaodhal_summary = None
    print('Skipped archive An Gaodhal reference training. Set RUN_ANGAODHAL_REFERENCE_TRAINING=1 only when intentionally reproducing the archived donor/reference branch.')


## Baseline Model — Target-Only TrOCR Fine-Tune


In [ ]:
# ## Main target TrOCR training run under trocr_target_only_{label}_* / train_summary output.
# Why: target-only baseline quantifies domain fit before transfer is considered.
target_run_dir = WORKSPACE.runs_dir / utc_run_id(f'trocr_target_only_{DATA_RUN_LABEL}')

target_summary = None
if RUN_TARGET_ONLY_TRAINING and target_rows:
    target_summary = train_trocr(
        manifest_path=target_manifest,
        output_dir=target_run_dir,
        model_name_or_path=MODEL_NAME,
        epochs=TRAINING_DEFAULTS['target_epochs'],
        batch_size=TRAINING_DEFAULTS['batch_size'],
        gradient_accumulation_steps=TRAINING_DEFAULTS['gradient_accumulation_steps'],
        learning_rate=TRAINING_DEFAULTS['learning_rate'],
        max_train_samples=TRAINING_DEFAULTS['max_train_samples'],
        max_val_samples=TRAINING_DEFAULTS['max_val_samples'],
        max_target_length=TRAINING_DEFAULTS['max_target_length'],
        warmup_ratio=TRAINING_DEFAULTS['warmup_ratio'],
        lr_scheduler_type=TRAINING_DEFAULTS['lr_scheduler_type'],
        device=TRAINING_DEFAULTS['device'],
        dataloader_num_workers=TRAINING_DEFAULTS['dataloader_num_workers'],
        dataloader_pin_memory=TRAINING_DEFAULTS['dataloader_pin_memory'],
        precision=TRAINING_DEFAULTS['precision'],
        run_name=target_run_dir.name,
        use_wandb=TRAINING_DEFAULTS['use_wandb'],
        wandb_project=TRAINING_DEFAULTS['wandb_project'],
        logging_steps=TRAINING_DEFAULTS['logging_steps'],
        max_optimizer_steps=TRAINING_DEFAULTS['max_optimizer_steps'],
        early_stopping_patience=TRAINING_DEFAULTS['early_stopping_patience'],
        early_stopping_min_delta=TRAINING_DEFAULTS['early_stopping_min_delta'],
        sanity_eval_after_n_evals=TRAINING_DEFAULTS['sanity_eval_after_n_evals'],
        sanity_max_eval_cer=TRAINING_DEFAULTS['sanity_max_eval_cer'],
        sanity_max_loss_multiple=TRAINING_DEFAULTS['sanity_max_loss_multiple'],
        wandb_alerts=TRAINING_DEFAULTS['wandb_alerts'],
        seed=TRAINING_DEFAULTS['seed'],
        allow_warm_start=ALLOW_CANONICAL_WARM_START,
    )
    training_summary_table(target_summary, 'Target Summary')
else:
    print('Skipped target-only training. RUN_TARGET_ONLY_TRAINING=', RUN_TARGET_ONLY_TRAINING, 'target rows=', len(target_rows))


## Canonical OCR Prediction Handoff For BART Controls

In [ ]:
# ## Resolve latest best checkpoint path for the canonical target-only run and export train/val/test predictions for BART controls.
def latest_target_checkpoint(workspace: Workspace, run_label: str) -> tuple[Path | None, str | None]:
    candidates = sorted(workspace.runs_dir.glob(f'trocr_target_only_{run_label}_*/*best_checkpoint'))
    if not candidates:
        candidates = sorted(workspace.runs_dir.glob('trocr_target_only_*/*best_checkpoint'))
    if not candidates:
        return None, None
    checkpoint = candidates[-1]
    return checkpoint, checkpoint.parent.name


canonical_prediction_dir = ensure_dir(WORKSPACE.runs_dir / 'ocr_training' / 'latest')
canonical_prediction_path = Path(os.environ.get('CANONICAL_TARGET_PREDICTIONS_PATH', str(canonical_prediction_dir / 'target_predictions.jsonl')))
canonical_prediction_summary_path = canonical_prediction_path.with_suffix('.summary.json')
prediction_manifest = Path(PREDICTION_MANIFEST_PATH) if PREDICTION_MANIFEST_PATH else target_manifest
prediction_rows = read_jsonl(prediction_manifest) if prediction_manifest.exists() else []
canonical_prediction_summary = None

if WRITE_CANONICAL_TARGET_PREDICTIONS and prediction_rows:
    override_checkpoint = os.environ.get('CANONICAL_PREDICTION_CHECKPOINT', '').strip()
    if override_checkpoint:
        prediction_checkpoint = Path(override_checkpoint)
        prediction_source_run_id = os.environ.get('CANONICAL_PREDICTION_SOURCE_RUN_ID', prediction_checkpoint.parent.name)
    elif target_summary:
        prediction_checkpoint = Path(target_summary['best_checkpoint'])
        prediction_source_run_id = target_summary['run_name']
    else:
        prediction_checkpoint, prediction_source_run_id = latest_target_checkpoint(WORKSPACE, DATA_RUN_LABEL)
    if prediction_checkpoint and prediction_checkpoint.exists():
        canonical_prediction_summary = infer_trocr_predictions(
            manifest_path=prediction_manifest,
            model_dir=prediction_checkpoint,
            output_path=canonical_prediction_path,
            splits=PREDICTION_SPLITS,
            batch_size=max(1, TRAINING_DEFAULTS['prediction_batch_size']),
            max_samples_per_split=TRAINING_DEFAULTS['max_predict_samples_per_split'],
            device=TRAINING_DEFAULTS['device'],
            max_target_length=TRAINING_DEFAULTS['max_target_length'],
            source_ocr_run_id=prediction_source_run_id,
        )
        write_json(canonical_prediction_summary_path, canonical_prediction_summary)
        summary_table(canonical_prediction_summary, 'Canonical OCR predictions')
    else:
        raise SystemExit('Cannot write canonical OCR predictions: no target-only or override best_checkpoint found.')
elif not prediction_rows:
    display(Markdown(f'Canonical prediction export skipped because no rows were loaded from `{prediction_manifest}`.'))
else:
    display(Markdown('Canonical prediction export disabled by WRITE_CANONICAL_TARGET_PREDICTIONS=0.'))


## Archive Only - Target Fine-Tune From An Gaodhal Checkpoint

In [ ]:
# ## Optional archival transfer run: start from the donor checkpoint only when reproducing the archived comparison.
# Why: the canonical target study centers on target-only OCR, while this branch remains available for documented reference reruns.
transfer_run_dir = WORKSPACE.runs_dir / utc_run_id(f'trocr_target_from_reference_{DATA_RUN_LABEL}')
transfer_source = os.environ.get('TRANSFER_SOURCE_CHECKPOINT', '').strip()
if transfer_source:
    TRANSFER_SOURCE_CHECKPOINT = Path(transfer_source)
elif angaodhal_summary:
    TRANSFER_SOURCE_CHECKPOINT = Path(angaodhal_summary['best_checkpoint'])
else:
    TRANSFER_SOURCE_CHECKPOINT = angaodhal_run_dir / 'best_checkpoint'

if RUN_TRANSFER_TRAINING and target_rows and TRANSFER_SOURCE_CHECKPOINT.exists():
    transfer_summary = train_trocr(
        manifest_path=target_manifest,
        output_dir=transfer_run_dir,
        model_name_or_path=str(TRANSFER_SOURCE_CHECKPOINT),
        epochs=TRAINING_DEFAULTS['transfer_epochs'],
        batch_size=TRAINING_DEFAULTS['batch_size'],
        gradient_accumulation_steps=TRAINING_DEFAULTS['gradient_accumulation_steps'],
        learning_rate=TRAINING_DEFAULTS['learning_rate'],
        max_train_samples=TRAINING_DEFAULTS['max_train_samples'],
        max_val_samples=TRAINING_DEFAULTS['max_val_samples'],
        max_target_length=TRAINING_DEFAULTS['max_target_length'],
        warmup_ratio=TRAINING_DEFAULTS['warmup_ratio'],
        lr_scheduler_type=TRAINING_DEFAULTS['lr_scheduler_type'],
        device=TRAINING_DEFAULTS['device'],
        dataloader_num_workers=TRAINING_DEFAULTS['dataloader_num_workers'],
        dataloader_pin_memory=TRAINING_DEFAULTS['dataloader_pin_memory'],
        precision=TRAINING_DEFAULTS['precision'],
        run_name=transfer_run_dir.name,
        use_wandb=TRAINING_DEFAULTS['use_wandb'],
        wandb_project=TRAINING_DEFAULTS['wandb_project'],
        logging_steps=TRAINING_DEFAULTS['logging_steps'],
        max_optimizer_steps=TRAINING_DEFAULTS['max_optimizer_steps'],
        early_stopping_patience=TRAINING_DEFAULTS['early_stopping_patience'],
        early_stopping_min_delta=TRAINING_DEFAULTS['early_stopping_min_delta'],
        sanity_eval_after_n_evals=TRAINING_DEFAULTS['sanity_eval_after_n_evals'],
        sanity_max_eval_cer=TRAINING_DEFAULTS['sanity_max_eval_cer'],
        sanity_max_loss_multiple=TRAINING_DEFAULTS['sanity_max_loss_multiple'],
        wandb_alerts=TRAINING_DEFAULTS['wandb_alerts'],
        seed=TRAINING_DEFAULTS['seed'],
        allow_warm_start=True,
    )
    training_summary_table(transfer_summary, 'Archive Archive Transfer Summary')
else:
    print('Skipped archive transfer run. RUN_TRANSFER_TRAINING=', RUN_TRANSFER_TRAINING, 'target rows=', len(target_rows), 'checkpoint exists=', TRANSFER_SOURCE_CHECKPOINT.exists())


## Plot Training Histories


In [ ]:
# ## Summarize all trocr_* run folders: pick latest per config, display training table.
import re

summary_paths = sorted(WORKSPACE.runs_dir.glob('trocr_*/*train_summary.json'))
run_names = [path.parent.name for path in summary_paths]
latest = {}
for run_name in run_names:
    config = re.sub(r"_\d{8}T\d{6}Z$", "", run_name)
    if config not in latest or run_name > latest[config]:
        latest[config] = run_name
runs_to_plot = list(latest.values())
summary_paths = [path for path in summary_paths if path.parent.name in runs_to_plot]
print('Found summaries:', len(summary_paths), '| plotting latest configs:', runs_to_plot)

history_rows = []
for path in summary_paths:
    summary = read_json(path)
    run_name = path.parent.name
    config_name = re.sub(r"_\d{8}T\d{6}Z$", "", run_name).replace('trocr_', '')
    for item in summary.get('history', []):
        history_rows.append({
            'run_name': run_name,
            'config_name': config_name,
            'epoch': item.get('epoch'),
            'train_loss': item.get('train_loss'),
            'val_cer': item.get('val_cer'),
            'learning_rate': item.get('learning_rate'),
            'model_name_or_path': summary.get('model_name_or_path'),
            'train_samples': summary.get('train_samples'),
            'val_samples': summary.get('val_samples'),
            'best_val_cer': summary.get('best_val_cer'),
            'lr_scheduler_type': summary.get('lr_scheduler_type'),
            'warmup_ratio': summary.get('warmup_ratio'),
        })

history_df = pd.DataFrame(history_rows)
if history_df.empty:
    display(Markdown('No training histories found yet. Run the P2 training cells, then rerun this cell to produce the learning curves.'))
else:
    display(history_df[['run_name', 'epoch', 'train_loss', 'val_cer', 'learning_rate', 'train_samples', 'val_samples']])
    ensure_dir(WORKSPACE.reports_dir)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    for config_name, run_df in history_df.groupby('config_name'):
        run_df = run_df.sort_values('epoch')
        label = config_name.replace('_', ' ')
        axes[0].plot(run_df['epoch'], run_df['train_loss'], marker='o', linewidth=2, label=label)
        axes[1].plot(run_df['epoch'], run_df['val_cer'], marker='o', linewidth=2, label=label)
        best = run_df.sort_values('val_cer').iloc[0]
        axes[1].annotate(f"best {best['val_cer']:.2f}", (best['epoch'], best['val_cer']), textcoords='offset points', xytext=(8, -14), fontsize=9)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-entropy loss')
    axes[0].grid(True, alpha=0.3)
    axes[1].set_title('Validation Error Rate (CER)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('CER, lower is better')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='best', fontsize=8)
    figure_path = WORKSPACE.reports_dir / 'training_metrics_overview.png'
    fig.suptitle('HGTCM TrOCR Learning Curves (latest run per config)', fontsize=14)
    fig.savefig(figure_path, dpi=180, bbox_inches='tight')
    plt.show()
    display(Markdown(f'Saved training metrics figure: `{figure_path}`'))

    best_rows = []
    for config_name, run_df in history_df.groupby('config_name'):
        best = run_df.sort_values('val_cer').iloc[0]
        best_rows.append({
            'config_name': config_name,
            'run_name': best['run_name'],
            'best_epoch': int(best['epoch']),
            'best_val_cer': float(best['val_cer']),
            'final_train_loss': float(run_df.sort_values('epoch').iloc[-1]['train_loss']),
            'model_name_or_path': best['model_name_or_path'],
            'lr_scheduler_type': best.get('lr_scheduler_type'),
            'warmup_ratio': best.get('warmup_ratio'),
        })
    display(pd.DataFrame(best_rows).sort_values('best_val_cer'))



## Part 2 Takeaways

Keep the canonical target-only artifacts for:

- the target-only TrOCR checkpoint
- the canonical train/val/test OCR prediction exports used by Part 3 BART controls
- the held-out raw OCR evaluation report for the scratch target model

The An Gaodhal reference and transfer branches remain archive-only and are intentionally excluded from the canonical target-study path unless explicitly re-enabled.